# Distillation desde el voting

Es el notebook que me dio el mejor resultado. La idea que me llevó a hacerlo: una vez que tengo un voting que combina DeBERTa-v3-large y Mistral-7B, sus predicciones sobre el test son una señal más fiable que las de cualquier modelo individual. Así que uso esas predicciones como pseudo-etiquetas y entreno un nuevo DeBERTa-large sobre train + test, dándole menos peso a las pseudo-etiquetas para que confíe más en lo real. Es pseudo-labeling, con un ensemble como maestro en lugar de un modelo individual.

In [ ]:
import os, gc, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LEN = 160
BATCH   = 8
GRAD_ACC = 2
EPOCHS  = 3
LR      = 1e-5
PSEUDO_WEIGHT = 0.7
SEED    = 42

TRAIN_PATH = '/kaggle/input/2025-26-false-political-claim-detection/train.csv'
TEST_PATH  = '/kaggle/input/2025-26-false-political-claim-detection/test_nolabel.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

voting = pd.read_csv('submission_voting_speaker_aware.csv')
test['label'] = voting['label'].values

Cargo los CSV originales y la submission del voting como pseudo-etiquetas del test. Elegí PSEUDO_WEIGHT = 0.7 después de probar 0.5, 0.7 y 1.0: con 1.0 el modelo se contagia demasiado de los errores del voting, con 0.5 la señal extra apenas se nota, 0.7 fue el punto medio que mejor me funcionó en OOF.

In [ ]:
def build_text(row):
    parts = []
    for col, prefix in [('speaker','Speaker'), ('party_affiliation','Party'),
                        ('speaker_job','Job'), ('state_info','State'),
                        ('subject','Topic')]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip() != '':
            parts.append(f'{prefix}: {v}')
    head = ' | '.join(parts)
    return f"{head} [SEP] Claim: {row['statement']}"

train['text_enriched'] = train.apply(build_text, axis=1)
test['text_enriched']  = test.apply(build_text, axis=1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_train = np.zeros(len(train), dtype=int)
for f, (_, vi) in enumerate(skf.split(train, train['label'])):
    folds_train[vi] = f

In [ ]:
train['weight'] = 1.0
train['fold']   = folds_train
test['weight']  = PSEUDO_WEIGHT
test['fold']    = -1

data = pd.concat([train[['text_enriched','label','weight','fold']],
                  test[['text_enriched','label','weight','fold']]], ignore_index=True)
y = data['label'].values.astype(int)
w = data['weight'].values.astype(np.float32)
folds_all = data['fold'].values
print(data.shape)

Los ejemplos del test entran al entrenamiento pero NUNCA al conjunto de validación. Por eso les asigno fold = -1: así en el bucle de folds 0..4 solo valido con etiquetas reales. Es importante para que el F1 OOF que reporto sea fiable y no esté inflado por evaluar contra las propias pseudo-etiquetas.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DistDs(Dataset):
    def __init__(self, texts, labels, weights):
        self.texts = texts; self.labels = labels; self.weights = weights
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN,
                        padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        item['weight'] = torch.tensor(self.weights[i], dtype=torch.float32)
        return item

class DebertaCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(MODEL_NAME)
        self.backbone.gradient_checkpointing_enable()
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(self.backbone.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
        return self.head(self.dropout(h))

In [ ]:
def train_fold(fold):
    tr_idx = np.where(folds_all != fold)[0]
    va_idx = np.where(folds_all == fold)[0]

    ds_tr = DistDs(data['text_enriched'].values[tr_idx], y[tr_idx], w[tr_idx])
    ds_va = DistDs(data['text_enriched'].values[va_idx], y[va_idx], w[va_idx])
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=2)
    dl_va = DataLoader(ds_va, batch_size=BATCH*2, shuffle=False, num_workers=2)

    model = DebertaCls().cuda()
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl_tr) * EPOCHS // GRAD_ACC
    sched = get_cosine_schedule_with_warmup(optim, int(0.1*total), total)
    class_w = torch.tensor([1.85, 1.0]).cuda()
    loss_fn = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05, reduction='none')

    best_f1, best_oof = 0, None
    for ep in range(EPOCHS):
        model.train()
        for step, batch in enumerate(dl_tr):
            ids = batch['input_ids'].cuda(); am = batch['attention_mask'].cuda()
            lab = batch['labels'].cuda(); wt = batch['weight'].cuda()
            logits = model(ids, am)
            loss_per_sample = loss_fn(logits, lab)
            loss = (loss_per_sample * wt).mean() / GRAD_ACC
            loss.backward()
            if (step+1) % GRAD_ACC == 0:
                optim.step(); sched.step(); optim.zero_grad()
        model.eval()
        probs = []
        with torch.no_grad():
            for batch in dl_va:
                logits = model(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
                probs.append(torch.softmax(logits,-1)[:,1].cpu().numpy())
        probs = np.concatenate(probs)
        f1 = f1_score(y[va_idx], (probs>0.5).astype(int), average='macro')
        print(f'fold {fold} ep {ep}  F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_oof = f1, probs
            torch.save(model.state_dict(), f'distill_fold{fold}.bin')
    return best_oof, va_idx

El truco clave: uso reduction='none' en CrossEntropy y multiplico la pérdida por el peso de cada muestra antes de promediar. Así las pseudo-etiquetas contribuyen un 70 % de lo que contribuiría una etiqueta real. Sin este reescalado, el test (que es casi la mitad del train en tamaño) dominaría el entrenamiento y el modelo se volcaría hacia los errores del voting.

In [ ]:
oof = np.zeros(len(train))
for f in range(5):
    np.random.seed(SEED+f); torch.manual_seed(SEED+f)
    oof_f, va_idx = train_fold(f)
    oof[va_idx] = oof_f
    gc.collect(); torch.cuda.empty_cache()

best_t, best_f1 = 0.5, 0
for t in np.arange(0.30, 0.70, 0.01):
    s = f1_score(train['label'].values, (oof>t).astype(int), average='macro')
    if s > best_f1: best_f1, best_t = s, t
print(f'OOF F1 = {best_f1:.4f}  thr {best_t:.2f}')

In [ ]:
ds_te = DistDs(test['text_enriched'].values,
               np.zeros(len(test), dtype=int),
               np.ones(len(test), dtype=np.float32))
dl_te = DataLoader(ds_te, batch_size=BATCH*2, shuffle=False, num_workers=2)
test_pred = np.zeros(len(test))
for f in range(5):
    model = DebertaCls().cuda()
    model.load_state_dict(torch.load(f'distill_fold{f}.bin'))
    model.eval()
    fold_probs = []
    with torch.no_grad():
        for batch in dl_te:
            logits = model(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
            fold_probs.append(torch.softmax(logits,-1)[:,1].cpu().numpy())
    test_pred += np.concatenate(fold_probs) / 5
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
np.save('oof_distill.npy', oof)
np.save('test_distill.npy', test_pred)

sub = pd.read_csv('/kaggle/input/2025-26-false-political-claim-detection/sample_submission.csv')
sub['label'] = (test_pred > best_t).astype(int)
sub.to_csv('submission_distill_voting.csv', index=False)
sub.head()

Esta es mi submission final. Combina lo mejor de las dos arquitecturas (encoder DeBERTa y LLM Mistral) por la vía del voting, y luego traslada ese consenso a un único modelo destilado más generalizable. 